# Creating CommonLID

In this notebook, I describe the combination of computational analysis and manual inspection used to transform the raw annotated data into the CommonLID evaluation dataset described in the paper.

In [7]:
import pandas as pd

## Initial filtering

The raw data has been normalised to have one tag per line. I read in the data as a data frame and inspect the data, then perform some basic cleaning.

In [19]:
# Login using e.g. `huggingface-cli login` to access this dataset
raw = pd.read_json(
    "hf://datasets/commoncrawl/CommonLID-raw/commonlid-raw-20251205.jsonl.gz", 
    lines=True
    )[["text", "tag", "line_id", "conflicts"]]
# removing columns related to converting to line-level annotations
raw

,text,tag,line_id,conflicts
0,Woo-woo? C’mon! It’s 2015!! | Jane Moody,aar,496145-c1973137-u1912-l0_000,[]
1,← The Benefits to You When a Healer Clears Her...,aar,496145-c1973137-u1912-l0_001,[]
2,Peulara binatang nakeuh buet,ace,472468-c814837-u1912-l0_000,[]
3,Bet-long lan mwen mové ki dé sèten moun | Mont...,acf,497311-c1380816-u9004-l0_000,[]
4,Envie de boycotter Israël ? Voici la liste des...,acf,497311-c1380816-u9004-l0_001,[]
...,...,...,...,...
541788,"20. Jika aku pengundi Kajang, aku akan pilih b...",zsm,489126-c765083-u9152-l0_018,[]
541789,21. Ini adalah masa paling sesuai untuk jadi c...,zsm,489126-c765083-u9152-l0_019,[]
541790,"22. Jika ada di antara anda yang opportunist, ...",zsm,489126-c765083-u9152-l0_020,[]
541791,"23. Jika pemimpin PAS bijak, letak calon anda....",zsm,489126-c765083-u9152-l0_021,[]


In [16]:
df = raw.copy()
print(f"before cleaning, there are {len(df):,} rows.")
df['text'] = df['text'].str.strip()
df = df.drop_duplicates(subset=["tag", "text"])
print(f"after stripping whitespace and deduplicating by tag and text, there are {len(df):,} rows.")
df = df[df["text"].str.len() > 10]
print(f"after filtering lines with length < 10, there are {len(df):,} rows.")
num_potential_conflicts = sum(df["conflicts"].apply(len)!= 0)
print(f"there are {num_potential_conflicts} lines which may have conflicts.")


before cleaning, there are 541,793 rows.
after stripping whitespace and deduplicating by tag and text, there are 382,419 rows.
after filtering lines with length < 10, there are 374,076 rows.
there are 3404 lines which may have conflicts.


## Checking for English

Manual inspection shows that there is a lot of English contamination, particular for lines with potential conflicts. I use the `langdetect` library as a basic check for mislabelled English. I skip five languages where there were too many false positives ('gla', 'gle', 'pcm', 'sot', 'sna').

In [20]:
# set up English detection
from langid.langid import LanguageIdentifier, model
identifier = LanguageIdentifier.from_modelstring(model, norm_probs=True)

def check_for_english(line):
    if line['tag'] in ['gla', 'gle', 'pcm', 'sot', 'sna']:  # special cases
        return None
    p_lang, score = identifier.classify(line['text'].lower())
    if p_lang == 'en' and line['tag']!='eng':  # mismatched tags
        if score > 0.7:
            return "highprob"
        if score < 0.2:
            return None
        else:
            return "lowerprob"

In [21]:
df.loc[:,'eng_check'] = df.apply(check_for_english, axis=1)
print(f"{sum(df["eng_check"]=="highprob")} rows are high probability English mismatch")
print(f"{sum(df["eng_check"]=="lowerprob")} rows have a lower-probability English mismatch")

5237 rows are high probability English mismatch
2020 rows have a lower-probability English mismatch


In [22]:
# let's look at some of the rows with high-probability English mismatch
df[df["eng_check"]=="highprob"].sample(20)

,text,tag,line_id,conflicts,eng_check
530959,Keywords Microgravity Electromagnetic Field Cu...,zho,502016-c879885-u9152-l0_104,[],highprob
207633,Beautiful 24-year-old lady discovers her colle...,hau,496615-c2072655-u9053-l0_021,[],highprob
387764,Imechapishwa na Information Unit Ministry of L...,swh,477253-c661970-u9067-l0_146,[],highprob
332536,"iaryylr March 1, 2011 at 3:47 AM",msa,497882-c1385231-u9152-l0_017,[],highprob
229409,"Posted by jenggot kambing on Wednesday, April ...",ind,490165-c761773-u9164-l0_020,[],highprob
288123,"↑ ""From 1863 to the Present Day"". FIFA.com. Di...",jav,491777-c725768-u9166-l0_009,[],highprob
305451,Home » Lahatsoratra farany farany » Vaovao Maf...,mlg,505239-c1372074-u9302-l0_001,[],highprob
517027,<P align=center><FONT size=4><FONT color=purpl...,vie,495965-c1113265-u9155-l0_066,[],highprob
473202,Oliy o’quv yurtlarida o’qitishning tashkiliy s...,uzb,499029-c1891604-u9242-l0_058,[],highprob
230133,Posted by jenggot kambing,ind,490210-c740580-u9166-l0_020,[],highprob


The sample shows that whilst there is some non-English, the majority of the text is actually English and/or low-quality boilerplate. Due to the scale, I decided to filter out all of the text detected as English with a high probability. Future work could be more discerning.

In [12]:
filtered = df[~(df["eng_check"] == "highprob")]
print(f"{len(df)-len(filtered)} rows likely mislabelled as non-English dropped, {len(filtered):,} rows remaining.")

5237 rows likely mislabelled as non-English dropped, 368,839 rows remaining.


## Manual inspection of potential conflicts

At this point, I begin manual analysis. Some of the remaining data has been marked up as having a potential conflict in labels: that is, the same span has received two or more different labels. I check through all of these manually to determine if they are simple mistakes or cases where multiple labels are valid. 

To do this, I sort the potential conflicts by text and tag, then compare identical lines with different tags. In the case of simple mistakes, I delete the label which is clearly misapplied. In the case where two or more labels may be valid, I keep the multiple copies of the line with a different label applied to each copy. 

I cannot replicate the manual analysis in this notebook, but I provide the code to split out the potential conflicts so you can look yourself. 

In [13]:
# I exported this as a text file, filtered through manually, then merged the remaining lines with the non-conflicts.
potential_conflicts = filtered[filtered['conflicts'].apply(len)!=0].sort_values(by=['text', 'tag'])

## Manual inspection of each language class

After merging in the manually-checked potential conflicts file with the lines without conflicts, the final step is to check a sample of each language class. If there are clear problems with a particular language class, I check more carefully. I'll demonstrate how to view a sample of each language class. For the actual analysis, you should do this with the version of the dataset after checking through all the potential conflicts.

In [14]:
# here is a set of all the language tags in the dataset
from pprint import pprint
pprint(f"{list(filtered['tag'].unique())}")

("['aar', 'ace', 'acf', 'fra', 'eng', 'gcr', 'spa', 'gcf', 'ara', 'aeb', "
 "'afr', 'amh', 'arb', 'ary', 'arg', 'ars', 'arz', 'apd', 'asm', 'aze', 'azj', "
 "'tur', 'rus', 'heb', 'bak', 'bcl', 'ben', 'bik', 'bre', 'ukr', 'ell', 'bul', "
 "'cat', 'ces', 'cmn', 'zho', 'yue', 'crh', 'pol', 'deu', 'ita', 'hau', 'vie', "
 "'est', 'ext', 'fas', 'fil', 'fin', 'fro', 'lat', 'fry', 'nld', 'fuv', 'gaz', "
 "'gla', 'gle', 'gom', 'hin', 'gug', 'guj', 'guw', 'hbo', 'gux', 'ibo', 'ind', "
 "'msa', 'jav', 'jpn', 'kab', 'kan', 'kik', 'kor', 'lav', 'ltg', 'lij', 'lug', "
 "'lvs', 'mal', 'mar', 'mlg', 'nso', 'nyn', 'oci', 'orm', 'ory', 'pan', 'pcm', "
 "'por', 'rcf', 'san', 'sna', 'zul', 'sot', 'swa', 'swh', 'tam', 'tat', 'tel', "
 "'tgl', 'tha', 'tuk', 'urd', 'uzb', 'uzs', 'vec', 'wuu', 'xho', 'yor', 'lin', "
 "'zsm']")


In [ ]:
tag_to_check = "gla"  # change me!
filtered[(filtered['tag'] == tag_to_check)][["text", "tag"]].sample(20)

,text,tag
190975,Chaochail an t-Oll. Henderson is e fhathast ai...,gla
190280,Foirm Autocapture - A ’taisbeanadh soidhnichea...,gla
190263,Nudgify: Meudaich na h-atharrachaidhean Shopif...,gla
190340,Thuirt luchd-labhairt gu bheil mòran aig a ’bh...,gla
190375,Bidh Emily Hennesey a ’toirt luchd-èisteachd a...,gla
191017,Chithear aithisg phàipeir-naidheachd mun fhacl...,gla
190537,"Nas fhaide dhen là, canaidh ceannard nan Làbar...",gla
191177,"Tha iasgachd cudromach sa sgìre, gu h-àraid ai...",gla
190515,"Tha an gealladh, a chaidh a shoidhneadh le Dai...",gla
190221,Tha tonna de stoidhlichean agus susbaint ann a...,gla


## CommonLID complete!

This notebook has taken you through the process used to create the CommonLID dataset. Any feedback or questions, please get in touch with the authors.